# Pipeline to streamline the following tasks

 1. Extract YT video urls to create a curated dataset  
 >[VIDEO_ID, VIDEO_TITLE, VIDEO_URL, Duration, Channel]  

 [2]. Extract audios from the videos using URLS or video_IDs.
 >Strip the .mp3 codec audio files from the videos and save them as a dataset.
 >> DataSet > VideoID > [audio.mp3]

 3. Establish process to extract / generate transcriptions from the videos.
 >Likely use of AI models for transcription due to no presence of youtube generated transcripts in the videos.

## Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import Modules

In [2]:
%%capture
!pip install yt-dlp -U
!apt-get update
!apt-get install -y ffmpeg

import pandas as pd
import yt_dlp

## Load and Filter DF

In [4]:
df = pd.read_csv('/content/drive/MyDrive/AnnamAI Tasks/YT_DATASET.csv')
print(f"Initial number of rows: {len(df)}")

def YT_Link_Generator(VIDEO_ID :str) -> str:
    """Generator Function to generate YT_Link from VIDEO_ID"""
    return "https://www.youtube.com/watch?v=" + VIDEO_ID

def exclude_shorts_and_Large(df):
    # omit videos which are less than 2 minutes (120 seconds) or larger than 2000 seconds
    condition = (df['Duration'] >= 120) & (df['Duration'] <= 2000)
    filtered_df = df[condition]
    residue_df = df[~condition]
    return filtered_df, residue_df

df_final, duration_residue_df = exclude_shorts_and_Large(df)
print(f"Rows after duration filtering: {len(df_final)}")
print(f"Rows removed by duration filtering: {len(duration_residue_df)}")

df = df_final

duration_residue_df.to_csv('/content/residue.csv', index=False)
print(f"Total rows in residue.csv: {len(duration_residue_df)}")


Initial number of rows: 13995
Rows after duration filtering: 12416
Rows removed by duration filtering: 1579
Total rows in residue.csv: 1579


In [5]:
df.head(10)

,YT_VIDEO_ID,YT_VIDEO_TITLE,YT_VIDEO_LINK,Duration,Channel
0,SxgaGXcQ8ZY,Krishi Darshan एकिकृत कृषि प्रणाली,https://www.youtube.com/watch?v=SxgaGXcQ8ZY,1377.0,Doordarshan National
1,JLwhLr4ZVd0,Krishi Darshan Bhagwani ke vikas ka harayana...,https://www.youtube.com/watch?v=JLwhLr4ZVd0,1472.0,Doordarshan National
2,ET8rwerJTg0,KRISHI DARSHAN GAON SAMRIDH BHART SAMRIDH,https://www.youtube.com/watch?v=ET8rwerJTg0,1491.0,Doordarshan National
3,BfMmAW-58ng,Krishi Darshan Chana Fasal,https://www.youtube.com/watch?v=BfMmAW-58ng,1221.0,Doordarshan National
4,VzSq2eW8ZG8,Krishi Darshan Zaed Phasal new,https://www.youtube.com/watch?v=VzSq2eW8ZG8,1476.0,Doordarshan National
5,tqu_7SftoxM,Suksham Sichai Se Badhyae Utpadan,https://www.youtube.com/watch?v=tqu_7SftoxM,1438.0,Doordarshan National
6,wlwZIC62hXQ,Gehu ki buaai,https://www.youtube.com/watch?v=wlwZIC62hXQ,1462.0,Doordarshan National
7,k3mJBSHc4m4,Krishi Darshan जायद की फसलों का कीटों से बचाव,https://www.youtube.com/watch?v=k3mJBSHc4m4,1472.0,Doordarshan National
8,_CrkOwCc4pU,"krishi darshan जैविक कृषि बाजार, दिल्ली हाट",https://www.youtube.com/watch?v=_CrkOwCc4pU,1538.0,Doordarshan National
9,5WlTrocLyL4,Sabjiyo mein kit prabandhan,https://www.youtube.com/watch?v=5WlTrocLyL4,1652.0,Doordarshan National


In [6]:
data_lists = {col: df[col].tolist() for col in df.columns}

# To verify, you can print the first few items of each list
for col_name, col_list in data_lists.items():
    print(f"Column '{col_name}': {col_list[:5]}...")

# Create separate list variables for each column from the data_lists dictionary
for col_name, col_list in data_lists.items():
    # Sanitize column names to be valid Python variable names and add '_list' suffix
    variable_name = col_name
    globals()[variable_name] = col_list

Column 'YT_VIDEO_ID': ['SxgaGXcQ8ZY', 'JLwhLr4ZVd0', 'ET8rwerJTg0', 'BfMmAW-58ng', 'VzSq2eW8ZG8']...
Column 'YT_VIDEO_TITLE': ['Krishi Darshan  एकिकृत कृषि प्रणाली', 'Krishi Darshan   Bhagwani ke vikas ka harayana me prayas', 'KRISHI DARSHAN GAON SAMRIDH BHART SAMRIDH', 'Krishi Darshan Chana Fasal', 'Krishi Darshan Zaed Phasal new']...
Column 'YT_VIDEO_LINK': ['https://www.youtube.com/watch?v=SxgaGXcQ8ZY', 'https://www.youtube.com/watch?v=JLwhLr4ZVd0', 'https://www.youtube.com/watch?v=ET8rwerJTg0', 'https://www.youtube.com/watch?v=BfMmAW-58ng', 'https://www.youtube.com/watch?v=VzSq2eW8ZG8']...
Column 'Duration': [1377.0, 1472.0, 1491.0, 1221.0, 1476.0]...
Column 'Channel': ['Doordarshan National', 'Doordarshan National', 'Doordarshan National', 'Doordarshan National', 'Doordarshan National']...


## DOWNLOAD AUDIO

In [7]:
#flag check for length
print(len(YT_VIDEO_ID) == len(YT_VIDEO_LINK) == len(YT_VIDEO_TITLE) == len(Duration) == len(Channel))

True


In [ ]:
# %%capture #Comment out if you dont want any outputs
def YT_Link_Generator(VIDEO_ID :str) -> str:
    """Generator Function to generate YT_Link from VIDEO_ID"""
    return "https://www.youtube.com/watch?v=" + VIDEO_ID

def Download_Audio(video_ID, video_Duration = 1800, channel = "ChannelName"):
    import yt_dlp # Import yt_dlp inside the function for multiprocessing

    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': f'/content/KrishiDarshan/{channel}/{video_ID}/audio_{video_Duration}.%(ext)s'
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([YT_Link_Generator(video_ID)])


# l = len(video_IDs)
# for i in range(l):
#     Don_Eladio(video_IDs[i], video_durations[i])


# l = len(YT_VIDEO_ID[:4])
# for i in range(l):
#     Download_Audio(YT_VIDEO_ID[i], Duration[i], Channel[i])


In [9]:
import multiprocessing as mp

inputs = list(zip(YT_VIDEO_ID, Duration, Channel))
print(len(inputs))

# Defined task : Don Eladio

if __name__ == '__main__':
    process_num = mp.cpu_count()
    print("TOTAL CORES: ", process_num)

    with mp.Pool(processes= process_num) as pool:
        pool.starmap(Download_Audio, inputs[:4])


    print("JOB FINISHED")


12416
TOTAL CORES:  2
[youtube] Extracting URL: https://www.youtube.com/watch?v=JLwhLr4ZVd0
[youtube] JLwhLr4ZVd0: Downloading webpage
[youtube] Extracting URL: https://www.youtube.com/watch?v=SxgaGXcQ8ZY
[youtube] SxgaGXcQ8ZY: Downloading webpage


[youtube] SxgaGXcQ8ZY: Downloading android vr player API JSON
[youtube] JLwhLr4ZVd0: Downloading android vr player API JSON
[info] SxgaGXcQ8ZY: Downloading 1 format(s): 140
[info] JLwhLr4ZVd0: Downloading 1 format(s): 140
[download] Destination: /content//KrishiDarshan/Doordarshan National/JLwhLr4ZVd0/audio_1472.0.m4a
[download]   1.1% of   22.29MiB at    9.80MiB/s ETA 00:02[download] Destination: /content//KrishiDarshan/Doordarshan National/SxgaGXcQ8ZY/audio_1377.0.m4a
[download] 100% of   22.29MiB in 00:00:02 at 9.63MiB/s   
[download] 100% of   20.85MiB in 00:00:02 at 7.59MiB/s   
[FixupM4a] Correcting container of "/content//KrishiDarshan/Doordarshan National/JLwhLr4ZVd0/audio_1472.0.m4a"
[FixupM4a] Correcting container of "/content//KrishiDarshan/Doordarshan National/SxgaGXcQ8ZY/audio_1377.0.m4a"
[ExtractAudio] Destination: /content//KrishiDarshan/Doordarshan National/SxgaGXcQ8ZY/audio_1377.0.wav
[ExtractAudio] Destination: /content//KrishiDarshan/Doordarshan National/JLwhLr4ZVd0/

[youtube] ET8rwerJTg0: Downloading android vr player API JSON


[youtube] BfMmAW-58ng: Downloading android vr player API JSON
[info] ET8rwerJTg0: Downloading 1 format(s): 140
[info] BfMmAW-58ng: Downloading 1 format(s): 140
[download] Destination: /content//KrishiDarshan/Doordarshan National/ET8rwerJTg0/audio_1491.0.m4a
[download]   8.7% of   23.01MiB at   11.14MiB/s ETA 00:01[download] Destination: /content//KrishiDarshan/Doordarshan National/BfMmAW-58ng/audio_1221.0.m4a
[download] 100% of   23.01MiB in 00:00:01 at 13.55MiB/s  
[FixupM4a] Correcting container of "/content//KrishiDarshan/Doordarshan National/ET8rwerJTg0/audio_1491.0.m4a"
[download] 100% of   18.83MiB in 00:00:01 at 9.52MiB/s   
[FixupM4a] Correcting container of "/content//KrishiDarshan/Doordarshan National/BfMmAW-58ng/audio_1221.0.m4a"
[ExtractAudio] Destination: /content//KrishiDarshan/Doordarshan National/ET8rwerJTg0/audio_1491.0.wav
[ExtractAudio] Destination: /content//KrishiDarshan/Doordarshan National/BfMmAW-58ng/audio_1221.0.wav
Deleting original file /content//KrishiDarsha

In [ ]:
#Zipping the files

!zip -r "/content/drive/MyDrive/AnnamAI Tasks/KrishiDarshan.zip" "/content/KrishiDarshan"